Box plots, price, sharpe, z testing for above normal performance
3. Use Sensitivity Analysis to check validity+ PNL analysis with change due to time, change due to market in addition to normal
need to add data normaliztion
5. Loop iteration ICIR testing

portmnanteau statistic, crossing statistic, box-tiao plot

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


Matplotlib is building the font cache; this may take a moment.


In [ ]:
from dataclasses import dataclass
from itertools import product
from typing import Callable


@dataclass(frozen=True)
class EvaluationConfig:
    date_col: str = "date"
    asset_col: str = "asset"
    signal_col: str = "signal"
    return_col: str = "forward_return"


def _safe_icir(ic_values: pd.Series) -> float:
    ic_values = ic_values.dropna()
    if len(ic_values) < 2:
        return np.nan
    std = ic_values.std(ddof=1)
    if std == 0 or np.isnan(std):
        return np.nan
    return ic_values.mean() / std


def _cross_sectional_ic(frame: pd.DataFrame, config: EvaluationConfig) -> float:
    subset = frame[[config.signal_col, config.return_col]].dropna()
    if len(subset) < 2:
        return np.nan
    return subset[config.signal_col].corr(subset[config.return_col], method="spearman")


def build_ic_series(panel: pd.DataFrame, config: EvaluationConfig) -> pd.Series:
    return (
        panel.groupby(config.date_col)
        .apply(lambda frame: _cross_sectional_ic(frame, config))
        .rename("ic")
    )


def evaluate_icir(panel: pd.DataFrame, config: EvaluationConfig = EvaluationConfig()) -> dict:
    ic_series = build_ic_series(panel, config)
    return {
        "ic_mean": ic_series.mean(),
        "ic_std": ic_series.std(ddof=1),
        "icir": _safe_icir(ic_series),
        "n_periods": int(ic_series.dropna().shape[0]),
    }


def run_rolling_icir_loop(
    panel: pd.DataFrame,
    strategy_fn: Callable[[pd.DataFrame, pd.DataFrame, dict], pd.DataFrame],
    parameter_grid: dict,
    config: EvaluationConfig = EvaluationConfig(),
    train_window: int = 252,
    test_window: int = 21,
) -> pd.DataFrame:
    """Evaluate a strategy over rolling windows and parameter combinations.

    strategy_fn must return a frame with at least the columns in config.signal_col
    and config.return_col for the supplied test slice.
    """
    if config.date_col not in panel.columns:
        raise ValueError(f"Missing required date column: {config.date_col}")

    ordered_dates = pd.Index(pd.to_datetime(panel[config.date_col]).sort_values().unique())
    param_names = list(parameter_grid.keys())
    param_values = list(parameter_grid.values())
    rows = []

    for start_idx in range(train_window, len(ordered_dates) - test_window + 1):
        train_dates = ordered_dates[start_idx - train_window : start_idx]
        test_dates = ordered_dates[start_idx : start_idx + test_window]
        train_slice = panel[panel[config.date_col].isin(train_dates)].copy()
        test_slice = panel[panel[config.date_col].isin(test_dates)].copy()

        for combo in product(*param_values):
            params = dict(zip(param_names, combo))
            evaluated = strategy_fn(train_slice, test_slice, params)
            ic_stats = evaluate_icir(evaluated, config)
            rows.append(
                {
                    "window_start": test_dates.min(),
                    "window_end": test_dates.max(),
                    **params,
                    **ic_stats,
                }
            )

    return pd.DataFrame(rows)


def summarize_sensitivity(results: pd.DataFrame, metric: str = "icir") -> pd.DataFrame:
    param_cols = [col for col in results.columns if col not in {"window_start", "window_end", "ic_mean", "ic_std", "icir", "n_periods"}]
    summary = (
        results.groupby(param_cols, dropna=False)[metric]
        .agg(["mean", "std", "count"])
        .reset_index()
        .sort_values("mean", ascending=False)
    )
    return summary


def plot_parameter_sensitivity(summary: pd.DataFrame, x: str, y: str, metric: str = "mean") -> None:
    pivot = summary.pivot(index=y, columns=x, values=metric)
    plt.figure(figsize=(10, 6))
    plt.imshow(pivot.values, aspect="auto", origin="lower", interpolation="nearest")
    plt.colorbar(label=metric)
    plt.xticks(range(len(pivot.columns)), list(pivot.columns), rotation=45, ha="right")
    plt.yticks(range(len(pivot.index)), list(pivot.index))
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f"Sensitivity of {metric}")
    plt.tight_layout()
    plt.show()


In [ ]:
def passthrough_strategy(_train_slice: pd.DataFrame, test_slice: pd.DataFrame, _params: dict) -> pd.DataFrame:
    """Example adapter for the ICIR loop.

    Replace this with the code that builds your model signals for the test slice.
    The returned frame must contain config.signal_col and config.return_col.
    """
    required_columns = {"signal", "forward_return"}
    missing = required_columns.difference(test_slice.columns)
    if missing:
        raise ValueError(f"Test slice is missing required columns: {sorted(missing)}")
    _ = (_train_slice, _params)
    return test_slice.copy()


parameter_grid = {
    "lookback": [20, 60, 120],
    "entry_z": [1.0, 1.5, 2.0],
    "cost_bps": [5, 10, 20],
}

# panel should be a long-form dataframe with columns:
# date, asset, signal, forward_return
# results = run_rolling_icir_loop(panel, passthrough_strategy, parameter_grid)
# summary = summarize_sensitivity(results, metric="icir")
# display(summary.head(10))
# plot_parameter_sensitivity(summary, x="entry_z", y="lookback", metric="mean")


### Transaction layer design

If you want to model execution realistically, keep the transaction layer separate from signal generation.

A clean setup is:

1. **Signal layer**: decides what to trade and when.
2. **Portfolio layer**: converts signals into target weights or target shares.
3. **Transaction layer**: turns targets into fills after costs, slippage, latency, partial fills, and borrow constraints.
4. **Ledger layer**: updates positions, cash, exposure, and realized/unrealized PnL.

The transaction layer should expose a function like `simulate_fill(order, market_state) -> fill_result`. That function can apply:

- fixed fees and commissions
- spread and slippage assumptions
- market impact based on trade size vs. liquidity
- short borrow fees and locate constraints
- partial fills if volume is low
- latency if you want the fill price to lag the signal time

In practice, you would pass the portfolio layer a **target position**, then let the transaction layer translate that target into an execution price and a realized fill quantity. The backtester should always mark performance on the filled position, not on the ideal target position.

For your stat arb setup, the key advantage is that you can run the same signal twice: once with a frictionless execution assumption, and once with a realistic transaction layer. The gap between the two is often the most useful diagnostic of whether the strategy survives live trading.
